# Tensor Parallelism in Needle: Autograd-Aware Collectives and Sharded Linear Layers

**10-414/714 Fall 2026 Final Project** · Evelyn Lui, Neethi Karnam, Anishka Mohanty

**Code:** TODO: private Google Drive link to the code zip (not a public GitHub repo)

> Scaffold notes (delete before submitting): every `TODO` marks prose or code still to write. Target length is about 2,000 words of prose. Cells marked **[live]** run in Colab. Cells marked **[results]** only load and plot files that the GHC runs produced.

## 0. Setup

Run these cells first. They fetch the code, install MPI, and put Needle on the import path.

In [ ]:
# Fetch the project code from the private Drive link.
# TODO: replace FILE_ID with the id from the shared zip's link.
import os
FILE_ID = "TODO"
if not os.path.exists("needle-tp"):
    !pip -q install gdown
    !gdown --id {FILE_ID} -O needle-tp.zip
    !unzip -q needle-tp.zip
CODE_DIR = os.path.abspath("needle-tp")
RESULTS_DIR = os.path.join(CODE_DIR, "results")   # files copied back from the GHC runs

In [ ]:
# MPI: Colab may not ship mpirun. mpi4py has to build against an installed MPI.
!which mpirun || (apt-get -qq update && apt-get -qq install -y openmpi-bin libopenmpi-dev)
!pip -q install mpi4py
!mpirun --version | head -1

In [ ]:
import sys
sys.path.insert(0, os.path.join(CODE_DIR, "python"))
os.environ.setdefault("NEEDLE_BACKEND", "nd")
import numpy as np
import needle as ndl

# Colab runs everything as root, and has fewer cores than ranks sometimes; mpirun needs both flags there.
MPIRUN = "mpirun --allow-run-as-root --oversubscribe"

## 1. Motivation

TODO (~250 words): why one GPU is not enough. Memory for weights + gradients + Adam state
(roughly 16 bytes per parameter in fp32). The goal: split each layer's weights across GPUs and
still train with the normal `.backward()`, matching a single-device run.

## 2. Background

TODO (~300 words): data vs. pipeline vs. tensor parallelism; what Megatron-LM does for the MLP
block (column-parallel then row-parallel); collectives (all-reduce = elementwise sum, every rank
gets the result). Cite:

- Shoeybi et al. 2019, *Megatron-LM: Training Multi-Billion Parameter Language Models Using Model Parallelism*, https://arxiv.org/abs/1909.08053
- NVIDIA Megatron-LM, `megatron/core/tensor_parallel/mappings.py`
- Hugging Face Picotron, `tensor_parallel.py`

## 3. Design

TODO (~400 words) + **our own diagram** (the plagiarism policy covers figures too, so no copied figures).

Layers of the system, bottom to top:

1. `needle.parallel.distributed`: `init`, `get_rank`, `get_world_size`, `all_reduce` over `mpi4py`
   (GPU → host → `Allreduce(SUM)` → same device). With no `mpirun` it runs as world size 1.
2. `needle.parallel.ops`: two `TensorOp`s, `CopyToParallelRegion` (f) and `ReduceFromParallelRegion` (g).
3. `needle.parallel.layers`: `ColumnParallelLinear`, `RowParallelLinear`.
4. Apps: the TP MLP block and the char-level language model.

What we did not change: `autograd.py` and `optim.py`. f and g are ordinary graph ops, and each
rank's optimizer updates only its own shards.

## 4. The math

TODO (~350 words). On rank $r$ of $N$:

$$Y = g\big(\mathrm{ReLU}(f(X)\,W_{1,r})\,W_{2,r}\big) + b_2$$

- $W_1$ is split by **columns**, so $\mathrm{ReLU}(X W_{1,r})$ is a slice: the final values of this rank's hidden units. No communication.
- $W_2$ is split by **rows**, so $Y_r$ is a full-shape **partial** that must be summed across ranks.

| Op | forward | backward |
|---|---|---|
| f = `CopyToParallelRegion` | identity | `all_reduce(out_grad)` |
| g = `ReduceFromParallelRegion` | `all_reduce(x)` | identity |

TODO: explain why g's backward is identity (the loss is replicated, so the gradient is already
complete) and why f's backward all-reduces (X fans out to every rank, so its adjoint is the sum of
the partial adjoints). Bias goes after g, or it is added N times.

The cell below simulates both ranks in one process with small integers, so the reader can check it by hand.

In [ ]:
# [live] Hand-checkable example: N = 2 ranks simulated in one process, plain numpy.
X  = np.array([[1, 2], [-1, 1]], dtype=np.float32)
W1 = np.array([[1, 0, -1, 2], [0, 1, 1, -1]], dtype=np.float32)
W2 = np.array([[3, 0], [0, 1], [1, 2], [2, -1]], dtype=np.float32)
N = 2

W1_shards = np.split(W1, N, axis=1)   # column-parallel
W2_shards = np.split(W2, N, axis=0)   # row-parallel
partials = [np.maximum(X @ A, 0) @ B for A, B in zip(W1_shards, W2_shards)]
for r, P in enumerate(partials):
    print(f"rank {r} partial Y_{r} =\n{P}")

Y_tp = sum(partials)                  # what g's all-reduce computes
Y_dense = np.maximum(X @ W1, 0) @ W2
print("sum of partials =\n", Y_tp)
print("dense          =\n", Y_dense)
print("max abs error  =", np.abs(Y_tp - Y_dense).max())

In [ ]:
# [live] Why f's backward must all-reduce: each rank only sees part of dL/dX.
# Take L = sum(Y), so dL/dY is all ones (replicated on every rank, hence g backward = identity).
dY = np.ones_like(Y_dense)
dX_parts = []
for A, B in zip(W1_shards, W2_shards):
    Z = X @ A
    dH = (dY @ B.T) * (Z > 0)          # back through row-parallel matmul and ReLU
    dX_parts.append(dH @ A.T)          # back through column-parallel matmul: a partial dX
dX_dense = ((dY @ W2.T) * (X @ W1 > 0)) @ W1.T
for r, p in enumerate(dX_parts):
    print(f"rank {r} partial dX =\n{p}")
print("sum (f's all-reduce) =\n", sum(dX_parts))
print("dense dX             =\n", dX_dense)

## 5. Correctness

TODO (~250 words): what we compare (forward output, every weight-shard gradient, dX, loss after
K optimizer steps) against a dense single-device Needle model with the same seed; tolerance (~1e-5)
and why results match only up to float rounding (the summation order differs).

The cell below launches **real separate processes** with `mpirun`. A notebook kernel is one
process, so MPI code has to run as a script.

In [ ]:
# [live] 2 ranks on Colab CPU.
# TODO: point at the real test script once it exists (milestone 1).
!cd {CODE_DIR} && {MPIRUN} -np 2 python tests_parallel/test_backward.py

## 6. Demo: char-level language model on Tiny Shakespeare

TODO (~250 words): model = last k chars → one-hot (k × 65) → ColumnParallelLinear → ReLU →
RowParallelLinear → logits (vocab ~65) → SoftmaxLoss. Same data seed on every rank. Output layer not sharded.

Short live run on CPU, then the full GHC run loaded from results.

In [ ]:
# [live] A few training steps, 2 ranks on CPU. TODO: script name and flags once written.
!cd {CODE_DIR} && {MPIRUN} -np 2 python apps/train_lm.py --steps 50 --out results/colab_tp.json

**Results file format** (written by rank 0; agree on this with whoever writes the scripts):

```json
// results/train_dense.json and results/train_tp.json
{"mode": "tp", "world_size": 2, "hardware": "GHC, 2x <GPU model>",
 "config": {"context": 8, "hidden": 4096, "lr": 0.001, "seed": 0},
 "step": [0, 10, 20], "loss": [4.17, 3.02, 2.71]}

// results/bench.json: one row per run
[{"world_size": 1, "hidden": 4096, "params_per_rank": 0, "peak_mem_mb": 0.0,
  "step_ms": 0.0, "allreduce_ms": 0.0, "host_copy_ms": 0.0}]
```

In [ ]:
# [results] Loss curves: dense single device vs. tensor-parallel (GHC run).
import json
import matplotlib.pyplot as plt

def load(name):
    path = os.path.join(RESULTS_DIR, name)
    if not os.path.exists(path):
        print(f"missing {path}: run it on GHC and copy it into results/")
        return None
    with open(path) as fh:
        return json.load(fh)

dense, tp = load("train_dense.json"), load("train_tp.json")
if dense and tp:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.plot(dense["step"], dense["loss"], label="dense, 1 GPU")
    ax.plot(tp["step"], tp["loss"], "--", label=f"tensor parallel, {tp['world_size']} GPUs")
    ax.set_xlabel("step"); ax.set_ylabel("training loss"); ax.legend()
    ax.set_title("Sharded training matches the dense model")
    plt.show()
    n = min(len(dense["loss"]), len(tp["loss"]))
    print("max |loss difference| =", np.abs(np.array(dense["loss"][:n]) - np.array(tp["loss"][:n])).max())

In [ ]:
# [results] Generated text (greedy decoding, same seed for dense and TP).
for name in ["samples_dense.txt", "samples_tp.txt"]:
    path = os.path.join(RESULTS_DIR, name)
    print(f"--- {name} ---")
    print(open(path).read() if os.path.exists(path) else "missing")

## 7. Benchmarks

TODO (~300 words): hardware (GHC machine, GPU model and memory, whether the 2 ranks are on one
machine or two); per-rank memory vs. model width; step time; communication cost including the
GPU → host → GPU copies MPI needs. Headline: a model that runs out of memory on one GPU trains on two.

In [ ]:
# [results] Per-rank memory and time vs. model width.
bench = load("bench.json")
if bench:
    import collections
    by_ws = collections.defaultdict(list)
    for row in sorted(bench, key=lambda r: r["hidden"]):
        by_ws[row["world_size"]].append(row)
    fig, (a, b) = plt.subplots(1, 2, figsize=(10, 3.5))
    for ws, rows in sorted(by_ws.items()):
        h = [r["hidden"] for r in rows]
        a.plot(h, [r["peak_mem_mb"] for r in rows], "o-", label=f"{ws} rank(s)")
        b.plot(h, [r["step_ms"] for r in rows], "o-", label=f"{ws} rank(s): step")
        if ws > 1:
            b.plot(h, [r["allreduce_ms"] + r["host_copy_ms"] for r in rows], "x:", label=f"{ws} ranks: comm")
    a.set_xlabel("hidden size h"); a.set_ylabel("peak memory per rank (MB)"); a.legend()
    b.set_xlabel("hidden size h"); b.set_ylabel("time per step (ms)"); b.legend()
    plt.tight_layout(); plt.show()

## 8. Limitations and future work

TODO (~150 words): host-side all-reduce cost and an NCCL backend behind the same interface;
attention-head and vocab parallelism; combining with data/pipeline parallelism.

## 9. Code

TODO: Drive link again, plus a short map of where everything lives
(`python/needle/parallel/`, `apps/`, `tests_parallel/`) and how to run it on a cluster:

```bash
cd needle-tp
mpirun -np 2 python apps/train_lm.py --out results/train_tp.json
```